In [1]:
import numpy as np

# -------------------------------------------------------
# Create a tiny toy dataset
# We want the network to learn: y = 2x + 1
# -------------------------------------------------------
x = np.array([[ -1.0],
              [ -0.5],
              [  0.0],
              [  0.5],
              [  1.0]])   # Input shape = (5 samples, 1 feature)

y = 2 * x + 1               # True outputs, same shape (5, 1)


In [2]:
# -------------------------------------------------------
# Network architecture
# - 1 input neuron
# - 2 hidden layer neurons (small to keep math simple)
# - 1 output neuron (linear regression)
# Hidden activation = sigmoid
# Loss function = Mean Squared Error
# -------------------------------------------------------
np.random.seed(0)           # for repeatable results

in_dim = 1                  # input dimension
hidden_dim = 2              # number of hidden neurons
out_dim = 1                 # output dimension


In [3]:
# -------------------------------------------------------
# Weight and bias initialization
# Small random weights + zero biases
# -------------------------------------------------------
W1 = np.random.randn(in_dim, hidden_dim) * 0.1   # shape (1, 2)
b1 = np.zeros((1, hidden_dim))                   # shape (1, 2)

W2 = np.random.randn(hidden_dim, out_dim) * 0.1  # shape (2, 1)
b2 = np.zeros((1, out_dim))                      # shape (1, 1)
'''
Every layer in a neural network has:
Weights (W) → numbers that decide how strongly inputs are connected
Biases (b) → constant offsets that help adjust the output
These are the parameters that the network learns during training.
Layer 1 (Hidden Layer)
Input dim = 1
Hidden neurons = 2
W1 = np.random.randn(1, 2) * 0.1
Imagine you are drawing a straight line:
y=2x+1
You already know how steep the line is (weights)
But the line needs to be moved up or down so it fits the data
That "+1" is the bias.
The training process discovers this value.
So we start with:
b2=0
and training adjusts it to:
b2≈1

because the correct equation is:

y=2x+1

'''

'\nEvery layer in a neural network has:\nWeights (W) → numbers that decide how strongly inputs are connected\nBiases (b) → constant offsets that help adjust the output\nThese are the parameters that the network learns during training.\nLayer 1 (Hidden Layer)\nInput dim = 1\nHidden neurons = 2\nW1 = np.random.randn(1, 2) * 0.1\nImagine you are drawing a straight line:\ny=2x+1\nYou already know how steep the line is (weights)\nBut the line needs to be moved up or down so it fits the data\nThat "+1" is the bias.\nThe training process discovers this value.\nSo we start with:\nb2=0\nand training adjusts it to:\nb2≈1\n\nbecause the correct equation is:\n\ny=2x+1\n\n'

In [4]:
# -------------------------------------------------------
# Sigmoid activation and derivative (for backprop)
# -------------------------------------------------------
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def sigmoid_prime(z):
    s = sigmoid(z)
    return s * (1 - s)
'''
Because backpropagation uses derivatives to compute gradients.
During backprop, we need:
∂𝐿/∂𝑧
for each neuron, where:
L = loss
z = weighted input to the neuron (before activation)
To compute this, we need the derivative of the activation function:

'''

'\nBecause backpropagation uses derivatives to compute gradients.\nDuring backprop, we need:\n∂𝐿/∂𝑧\nfor each neuron, where:\nL = loss\nz = weighted input to the neuron (before activation)\nTo compute this, we need the derivative of the activation function:\n\n'

In [5]:
# -------------------------------------------------------
# Hyperparameters
# -------------------------------------------------------
lr = 0.1         # learning rate
epochs = 200     # training iterations



In [6]:
# -------------------------------------------------------
# Mean squared error
# -------------------------------------------------------
def mse(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)


In [7]:
# -------------------------------------------------------
# TRAINING LOOP — Forward + Backward Pass
# -------------------------------------------------------
for epoch in range(1, epochs + 1):

    # ============================
    # 1) FORWARD PASS
    # ============================

    # Hidden layer linear combination: z1 = xW1 + b1
    z1 = x @ W1 + b1                   # shape (N, 2)

    # Apply sigmoid activation
    a1 = sigmoid(z1)                   # shape (N, 2)

    # Output layer: z2 = a1W2 + b2  (linear output)
    z2 = a1 @ W2 + b2                  # shape (N, 1)
    y_pred = z2                        # final output


    # ============================
    # 2) COMPUTE LOSS
    # ============================
    loss = mse(y, y_pred)


    # ============================
    # 3) BACKWARD PASS
    # ============================
    N = x.shape[0]                     # number of samples
    #x=[rows,columns] x[0]=no of rows
    # Gradient of MSE w.r.t. predictions
    # dL/dy_pred = (2/N)*(y_pred - y)
    dL_dy = (2.0 / N) * (y_pred - y)   # shape (N, 1)
    '''
    y_pred - y = "How wrong was the model?"
    Multiply by (2/N) = scaling factor for MSE
    The result (dL_dy) tells us how to adjust weights in the backward pass.
    '''
    # --------------------------------
    # Gradients for OUTPUT layer
    # --------------------------------
    # y_pred = a1W2 + b2
    # dW2 = a1^T * dL_dy
    dW2 = a1.T @ dL_dy                 # shape (2, 1)
    
    # db2 = sum of gradients for each sample
    db2 = np.sum(dL_dy, axis=0, keepdims=True)   # shape (1, 1)
    #How much should the output bias b2 be updated?”

    # --------------------------------
    # Backpropagate into hidden layer
    # --------------------------------
    # delta_hidden = dL/dz1
    # = (dL/dy_pred @ W2^T) * sigmoid'(z1)
    delta_hidden = (dL_dy @ W2.T) * sigmoid_prime(z1)   # shape (N, 2)

    # Gradients for W1 and b1
    # dW1 = x^T @ delta_hidden
    dW1 = x.T @ delta_hidden              # shape (1, 2)

    # db1 = row-wise sum
    db1 = np.sum(delta_hidden, axis=0, keepdims=True)   # shape (1, 2)


    # ============================
    # 4) UPDATE WEIGHTS
    # ============================
    W2 -= lr * dW2
    b2 -= lr * db2

    W1 -= lr * dW1
    b1 -= lr * db1


    # Print loss occasionally
    if epoch % 20 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d}  Loss = {loss:.6f}")
    #This code prints the loss value every 20 epochs 
    #so you can see how well the model is learning without printing too much text.

# -------------------------------------------------------
# After training: show predictions
# -------------------------------------------------------
print("\nFinal predictions vs true values:")

# Final forward pass
'''
Runs the neural network one last time after training.
Calculates the final predictions using the learned weights.
Prints each input value, the true output, and the predicted output.
Helps you see if the model learned the function correctly.
'''
z1 = x @ W1 + b1
a1 = sigmoid(z1)
y_pred = a1 @ W2 + b2

for xi, yi, ypi in zip(x.flatten(), y.flatten(), y_pred.flatten()):
    print(f"x={xi: .2f}  true={yi: .2f}  pred={ypi: .3f}")


Epoch   1  Loss = 2.690877
Epoch  20  Loss = 1.697725
Epoch  40  Loss = 1.064266
Epoch  60  Loss = 0.426309
Epoch  80  Loss = 0.137955
Epoch 100  Loss = 0.051928
Epoch 120  Loss = 0.029848
Epoch 140  Loss = 0.024010
Epoch 160  Loss = 0.022001
Epoch 180  Loss = 0.020921
Epoch 200  Loss = 0.020115

Final predictions vs true values:
x=-1.00  true=-1.00  pred=-0.806
x=-0.50  true= 0.00  pred=-0.175
x= 0.00  true= 1.00  pred= 0.942
x= 0.50  true= 2.00  pred= 2.148
x= 1.00  true= 3.00  pred= 2.919
